# Intro

### Compile for the CPU

In [1]:
from numba import jit
import math

In [2]:
@jit
def hypot(x, y):
    x = abs(x);
    y = abs(y);
    t = min(x, y);
    x = max(x, y);
    t = t / x;
    return x * math.sqrt(1+t*t)

In [3]:
hypot(3.0, 4.0)

5.0

Numba also saves the original Python implementation of the function in the .py_func attribute.

In [4]:
hypot.py_func(3.0, 4.0)

5.0

#### benchmark python vs. numba-jit

In [5]:
%timeit hypot.py_func(3.0, 4.0)

489 ns ± 166 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [6]:
%timeit hypot(3.0, 4.0)

157 ns ± 2.58 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


type inference

In [7]:
hypot.inspect_types()

hypot (float64, float64)
--------------------------------------------------------------------------------
# File: /tmp/ipykernel_39667/2042191135.py
# --- LINE 1 --- 
# label 0
#   x = arg(0, name=x)  :: float64
#   y = arg(1, name=y)  :: float64

@jit

# --- LINE 2 --- 

def hypot(x, y):

    # --- LINE 3 --- 
    #   $4load_global.0 = global(abs: <built-in function abs>)  :: Function(<built-in function abs>)
    #   x.1 = call $4load_global.0(x, func=$4load_global.0, args=[Var(x, 2042191135.py:1)], kws=(), vararg=None, varkwarg=None, target=None)  :: (float64,) -> float64
    #   del x
    #   del $4load_global.0

    x = abs(x);

    # --- LINE 4 --- 
    #   $26load_global.4 = global(abs: <built-in function abs>)  :: Function(<built-in function abs>)
    #   y.1 = call $26load_global.4(y, func=$26load_global.4, args=[Var(y, 2042191135.py:1)], kws=(), vararg=None, varkwarg=None, target=None)  :: (float64,) -> float64
    #   del y
    #   del $26load_global.4

    y = abs(y);

    #

## NumPy Universal Functions (ufuncs)

Examples:

In [8]:
import numpy as np

a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

np.add(a, b) 

array([11, 22, 33, 44])

In [9]:
np.add(a, 100)

array([101, 102, 103, 104])

In [10]:
c = np.arange(4*4).reshape((4,4))
print('c:', c)

np.add(b, c)

c: [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]


array([[10, 21, 32, 43],
       [14, 25, 36, 47],
       [18, 29, 40, 51],
       [22, 33, 44, 55]])

#### Making ufuncs for the GPU

In [12]:
from numba import vectorize

In [17]:
import warnings
warnings.filterwarnings('ignore')

In [13]:
@vectorize
def add_ten(num):
    return num + 10

In [14]:
nums = np.arange(10)
add_ten(nums) # pass the whole array into the ufunc, it performs the operation on each element

array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19])

In [15]:
@vectorize(['int64(int64, int64)'], target='cuda') # Type signature and target are required for the GPU
def add_ufunc(x, y):
    return x + y

In [18]:
add_ufunc(a, b)

array([11, 22, 33, 44])

#### benchmark numpy-cpu vs. numba-gpu

In [19]:
%timeit np.add(b, c)   # NumPy on CPU

1.06 μs ± 88 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [20]:
%timeit add_ufunc(b, c) # Numba on GPU

691 μs ± 64.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


GPU is underutilized 👆️

In [21]:
import math # Note that for the CUDA target, we need to use the scalar functions from the math module, not NumPy

SQRT_2PI = np.float32((2*math.pi)**0.5)  # Precompute this constant as a float32.  Numba will inline it at compile time.

@vectorize(['float32(float32, float32, float32)'], target='cuda')
def gaussian_pdf(x, mean, sigma):
    '''Compute the value of a Gaussian probability density function at x with given mean and sigma.'''
    return math.exp(-0.5 * ((x - mean) / sigma)**2) / (sigma * SQRT_2PI)

In [ ]:
# Evaluate the Gaussian a million times!
x = np.random.uniform(-3, 3, size=1000000).astype(np.float32)
mean = np.float32(0.0)
sigma = np.float32(1.0)

# Quick test on a single element just to make sure it works
gaussian_pdf(x[0], 0.0, 1.0)

array([0.00649193], dtype=float32)

In [24]:
%timeit gaussian_pdf(x, mean, sigma)

6.37 ms ± 54.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
@vectorize
def cpu_gaussian_pdf(x, mean, sigma):
    '''Compute the value of a Gaussian probability density function at x with given mean and sigma.'''
    return math.exp(-0.5 * ((x - mean) / sigma)**2) / (sigma * SQRT_2PI)

In [26]:
%timeit cpu_gaussian_pdf(x, mean, sigma)

13.1 ms ± 4.52 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


## CUDA Device Functions

To compile functions for the GPU that are not element wise, vectorized functions, we use numba.cuda.jit.

In [28]:
from numba import cuda

@cuda.jit(device=True)
def polar_to_cartesian(rho, theta):
    x = rho * math.cos(theta)
    y = rho * math.sin(theta)
    return x, y

@vectorize(['float32(float32, float32, float32, float32)'], target='cuda')
def polar_distance(rho1, theta1, rho2, theta2):
    x1, y1 = polar_to_cartesian(rho1, theta1) # We can use device functions inside our GPU ufuncs
    x2, y2 = polar_to_cartesian(rho2, theta2)
    
    return ((x1 - x2)**2 + (y1 - y2)**2)**0.5

In [29]:
n = 1000000
rho1 = np.random.uniform(0.5, 1.5, size=n).astype(np.float32)
theta1 = np.random.uniform(-np.pi, np.pi, size=n).astype(np.float32)
rho2 = np.random.uniform(0.5, 1.5, size=n).astype(np.float32)
theta2 = np.random.uniform(-np.pi, np.pi, size=n).astype(np.float32)

In [30]:
polar_distance(rho1, theta1, rho2, theta2)

array([2.7705188 , 2.3055646 , 1.1751504 , ..., 0.92688096, 2.4845932 ,
       0.8331757 ], shape=(1000000,), dtype=float32)

## Managing GPU Memory

In [31]:
@vectorize(['float32(float32, float32)'], target='cuda')
def add_ufunc(x, y):
    return x + y

In [32]:
n = 100000
x = np.arange(n).astype(np.float32)
y = 2 * x

In [33]:
%timeit add_ufunc(x, y)  # Baseline performance with host arrays

1.05 ms ± 73 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [34]:
x_device = cuda.to_device(x)
y_device = cuda.to_device(y)

print(x_device)
print(x_device.shape)
print(x_device.dtype)

(100000,)
float32


In [35]:
%timeit add_ufunc(x_device, y_device)

288 μs ± 3.09 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [36]:
out_device = cuda.device_array(shape=(n,), dtype=np.float32)

In [37]:
%timeit add_ufunc(x_device, y_device, out=out_device)

166 μs ± 7.77 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [38]:
out_host = out_device.copy_to_host()
print(out_host[:10])

[ 0.  3.  6.  9. 12. 15. 18. 21. 24. 27.]
